# 3.5 — Making the block power method cheaper

The block power method `block_transfer_eigs` is the workhorse for the non-integrable case: it is the
only route that stays converged through the transfer-matrix gap closing (notebook 3) and so feeds
both the entropy profiles and the leading-eigenvalue central charge of notebook 7. But it is
**expensive** — the notebook-7 eigenvalue sweep alone ran for hours. This notebook is a focused
engineering study: profile where the cost goes, then benchmark **every lever** that could make it
cheaper, and conclude with a recommended configuration.

**Reference point.** All benchmarks use a single hard configuration — Alcaraz $p=0.1$, $T=6$,
`nbeta=4` — chosen because it sits in the gap-closing regime on an already-long temporal chain.
Everything is cached to `results/data/nb35_blockpm_bench.jld2`, keyed by config, so the notebook
re-opens without re-running the heavy cells. **These cells are heavy; run them deliberately.**

In [ ]:
include("src/thesislib.jl")
using Printf, LsqFit

# Shared reference tMPO (built once, reused across all configs).
P_REF, T_REF, NBETA_REF = 0.1, 6.0, 4
mpo_ref, scaf_ref = build_alcaraz_tmpo(T_REF; p=P_REF, lambda=1.0, dt=0.1, nbeta=NBETA_REF, MPO_alg="VD2")
@info "reference tMPO: T=$T_REF, $(length(scaf_ref)) time sites, temporal phys dim = $(dim(siteinds(scaf_ref)[1]))"

# Cache of benchmark records, keyed by an auto-generated config string (identical configs dedupe).
BENCH_FILE  = "results/data/nb35_blockpm_bench.jld2"
bench_cache = isfile(BENCH_FILE) ? load(BENCH_FILE, "cache") : Dict{String,Any}()

function bench!(; k=4, maxdim=256, cutoff=1e-12, itermax=200, eps_conv=1e-6, maxdims=nothing,
                  mpo=mpo_ref, scaf=scaf_ref, tag="")
    sched = maxdims === nothing ? "fix" : "ramp$(first(maxdims))-$(last(maxdims))L$(length(maxdims))"
    key = "k$(k)_md$(maxdim)_cut$(cutoff)_it$(itermax)_$(sched)" * (isempty(tag) ? "" : "_$tag")
    haskey(bench_cache, key) && return bench_cache[key]
    t = @timed block_transfer_eigs(mpo, scaf; k=k, maxdim=maxdim, cutoff=cutoff,
                                   itermax=itermax, eps_conv=eps_conv, maxdims=maxdims)
    theta, L, R, info = t.value
    rec = (key=key, time=t.time, theta=collect(theta), niters=info[:niters],
           reason=string(info[:reason]), chi=maxlinkdim(R[1]))
    bench_cache[key] = rec; jldsave(BENCH_FILE; cache=bench_cache)
    @info "$key  t=$(round(t.time,digits=1))s  it=$(info[:niters])  $(info[:reason])  χ=$(rec.chi)  |θ0|=$(round(abs(theta[1]),digits=5))"
    return rec
end

# Accurate reference eigenvalues (everything else is compared to these).
ref = bench!(k=4, maxdim=256, cutoff=1e-12, itermax=300)
theta_ref = ref.theta
dθ0(r) = abs(r.theta[1]-theta_ref[1]); dθ1(r) = abs(r.theta[2]-theta_ref[2])
@printf("REFERENCE  |θ0|=%.6f  |θ1|=%.6f   (%d iters, %s, %.1fs, χ=%d)\n",
        abs(theta_ref[1]), abs(theta_ref[2]), ref.niters, ref.reason, ref.time, ref.chi)

[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
┌ Warning: Tensor *not* symmetric (dim=2|id=629|"S=1/2,Site") <-> (dim=2|id=629|"S=1/2,Site")', normdiff = 0.2034249757191385
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on bond(space) => phys(time) indices
┌ Warning: Tensor *not* symmetric (dim=7|id=483|"Link,l=1") <-> (dim=7|id=627|"Link,l=2"), normdiff = 1.1566412865695994
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
┌ Warning: Tensor *not* symmetric (dim=2|id=520|"S=1/2,Site") <-> (dim=2|id=520|"S=1/2,Site")', normdiff = 0.2034249757191385
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93


## Where the cost goes

Each iteration of `block_transfer_eigs` does the expensive work in two places (see
`src/transverse_tools.jl`): the **$2k$ `applyn` calls** that apply the tMPO to every Ritz vector
(lines ~82–83) and the **$2k$ `lincomb_mps` de-mixings** that rotate the block by the Ritz
coefficients (lines ~120–121). The $k\times k$ pencil solve (`pinv`, `eigen`) is negligible ($k\le4$).
Both expensive steps scale with the **temporal chain length** and with $\chi^2$ (the truncation bond
dim). The single most telling number is below: the **converged bond dim vs the `maxdim=256` cap** —
because the temporal entanglement grows only logarithmically, the converged $\chi$ is typically far
below the cap, which means most of the truncation budget (and time) is wasted.

In [ ]:
# The converged bond dim vs the cap — is maxdim=256 oversized?
@printf("converged χ (R[1]) = %d   vs   maxdim cap = 256   ⇒  cap is %s\n",
        ref.chi, ref.chi < 200 ? "OVERSIZED (room to shrink)" : "near the cap")
@printf("per-iteration wall time ≈ %.2f s/iter  (%.1fs over %d iters)\n",
        ref.time/max(ref.niters,1), ref.time, ref.niters)

## Route A — the maxdim cap

If the converged $\chi$ is well below 256, capping `maxdim` lower should cut time with no loss of
$\lambda_0,\lambda_1$. We sweep the cap and watch both wall-time and the eigenvalue error vs the
reference.

In [ ]:
mds = [48, 64, 96, 128, 256]
for md in mds; bench!(k=4, maxdim=md, cutoff=1e-12, itermax=300); end
recsA = [bench_cache["k4_md$(md)_cut1.0e-12_it300_fix"] for md in mds]
@printf("%-7s %-9s %-8s %-10s %-10s %-7s %-9s\n","maxdim","time(s)","niters","|Δθ0|","|Δθ1|","χ","reason")
for (md,r) in zip(mds, recsA)
    @printf("%-7d %-9.1f %-8d %-10.2e %-10.2e %-7d %-9s\n", md, r.time, r.niters, dθ0(r), dθ1(r), r.chi, r.reason)
end
pt = plot(mds, [r.time for r in recsA]; xlabel="maxdim", ylabel="wall time (s)", marker=:circle, lw=2,
          label="time", framestyle=:box, grid=true, title="Route A: cost vs maxdim cap")
pe = plot(mds, [max(dθ0(r),1e-16) for r in recsA]; xlabel="maxdim", ylabel="|Δθ| vs ref", yscale=:log10,
          marker=:circle, lw=2, label="|Δθ0|", framestyle=:box, grid=true, title="Eigenvalue error")
plot!(pe, mds, [max(dθ1(r),1e-16) for r in recsA]; marker=:square, lw=2, label="|Δθ1|")
plot(pt, pe; layout=(1,2), size=(1100,420), margin=5Plots.mm)

## Route B — a maxdim ramp (schedule)

Early iterations don't need the full bond dim — the Ritz vectors are still far from converged. The
new `maxdims` kwarg (added to `block_transfer_eigs`, backward-compatible: `nothing` ⇒ the fixed
`maxdim`) ramps $\chi$ from small to the cap, making the early iterations cheap. We compare a ramp to
the fixed cap at the same final bond dim.

In [ ]:
ramp = round.(Int, range(16, 128, length=40))   # 16 → 128 over 40 iters, then holds 128
rB_fix  = bench!(k=4, maxdim=128, cutoff=1e-12, itermax=300)                 # fixed 128 (Route A)
rB_ramp = bench!(k=4, maxdim=128, cutoff=1e-12, itermax=300, maxdims=ramp)   # ramped 16→128
@printf("%-14s %-9s %-8s %-10s %-10s %s\n","schedule","time(s)","niters","|Δθ0|","|Δθ1|","reason")
@printf("%-14s %-9.1f %-8d %-10.2e %-10.2e %s\n","fixed 128", rB_fix.time, rB_fix.niters, dθ0(rB_fix), dθ1(rB_fix), rB_fix.reason)
@printf("%-14s %-9.1f %-8d %-10.2e %-10.2e %s\n","ramp 16→128", rB_ramp.time, rB_ramp.niters, dθ0(rB_ramp), dθ1(rB_ramp), rB_ramp.reason)

## Route C — the truncation cutoff

Eigenvalues are robust to a looser SVD cutoff (it truncates more aggressively ⇒ smaller effective
$\chi$ ⇒ faster). We sweep the cutoff at a fixed moderate cap and check the eigenvalues hold.

In [ ]:
for ct in [1e-12, 1e-10, 1e-8]; bench!(k=4, maxdim=128, cutoff=ct, itermax=300); end
@printf("%-9s %-9s %-8s %-10s %-10s %-7s %s\n","cutoff","time(s)","niters","|Δθ0|","|Δθ1|","χ","reason")
for ct in [1e-12, 1e-10, 1e-8]
    r = bench_cache["k4_md128_cut$(ct)_it300_fix"]
    @printf("%-9.0e %-9.1f %-8d %-10.2e %-10.2e %-7d %s\n", ct, r.time, r.niters, dθ0(r), dθ1(r), r.chi, r.reason)
end

## Route D — the exponentiation kernel (WII vs VD2)

After the $90^\circ$ rotation the spatial MPO's virtual bond becomes the temporal **physical**
dimension: $1+\chi+\chi^2$ for VD2 vs $1+\chi$ for WII (notebook 2). A smaller temporal physical
dimension makes every `applyn` cheaper. WII is only 1st order for this NNN model, so it may bias the
*absolute* $\lambda$ — but the convention-free headline (the $p{=}0.1/p{=}0$ ratio) could still
survive. Here we just check the eigenvalues and the cost at $p=0.1$; the ratio check belongs to the
production sweep.

In [ ]:
mpo_wii, scaf_wii = build_alcaraz_tmpo(T_REF; p=P_REF, lambda=1.0, dt=0.1, nbeta=NBETA_REF, MPO_alg="WII")
@info "WII temporal phys dim = $(dim(siteinds(scaf_wii)[1]))  (VD2 was $(dim(siteinds(scaf_ref)[1])))"
rD = bench!(k=4, maxdim=128, cutoff=1e-10, itermax=300, mpo=mpo_wii, scaf=scaf_wii, tag="WII")
rD_vd2 = bench_cache["k4_md128_cut1.0e-10_it300_fix"]   # VD2 counterpart from Route C
@printf("%-6s %-9s %-8s %-12s %-12s %s\n","kernel","time(s)","niters","|θ0|","|θ1|","reason")
@printf("%-6s %-9.1f %-8d %-12.6f %-12.6f %s\n","VD2", rD_vd2.time, rD_vd2.niters, abs(rD_vd2.theta[1]), abs(rD_vd2.theta[2]), rD_vd2.reason)
@printf("%-6s %-9.1f %-8d %-12.6f %-12.6f %s\n","WII", rD.time, rD.niters, abs(rD.theta[1]), abs(rD.theta[2]), rD.reason)

## Route E — warm starts

`block_transfer_eigs` now accepts `seedL`/`seedR` (backward-compatible: `nothing` ⇒ random). A
converged pair reused as the seed should converge almost immediately. Below we prove the mechanism at
the **same** $(p,T)$: a cold (random) run, then a warm run seeded with its output.

The real production win is **cross-$T$** warm starting: the temporal MPS for $T+\Delta T$ is nearly
the $T$ one with extra bulk sites, so seeding the $T+\Delta T$ sweep from the (extended) converged
$T$ vectors should slash the iteration count along a $T$-ladder. That needs a `pad_tmps` helper to
move the converged tensors onto the longer scaffold's site indices — sketched as the highest-payoff
follow-up below.

In [ ]:
# Same-(p,T) warm start: random cold run, then re-seed with its converged vectors.
t1 = @timed block_transfer_eigs(mpo_ref, scaf_ref; k=2, maxdim=128, cutoff=1e-10, itermax=300, eps_conv=1e-6)
th1, L1, R1, info1 = t1.value
t2 = @timed block_transfer_eigs(mpo_ref, scaf_ref; k=2, maxdim=128, cutoff=1e-10, itermax=300, eps_conv=1e-6,
                                seedL=L1, seedR=R1)
th2, L2, R2, info2 = t2.value
@printf("cold (random seed):  %3d iters  %.1fs   |θ0|=%.6f\n", info1[:niters], t1.time, abs(th1[1]))
@printf("warm (reuse output): %3d iters  %.1fs   |θ0|=%.6f   Δθ0=%.2e\n",
        info2[:niters], t2.time, abs(th2[1]), abs(th2[1]-th1[1]))

## Route F — block size $k$ and de-mixing

Route 2 (eigenvalues) needs only $\lambda_0,\lambda_1$, so $k=2$ is the minimum; $k=4$ costs twice
the `applyn` work per iteration but can de-mix the leading pair more cleanly near the degeneracy. We
compare $k=2$ vs $k=4$ at the otherwise-cheap config.

In [ ]:
rF2 = bench!(k=2, maxdim=128, cutoff=1e-10, itermax=300)
rF4 = bench_cache["k4_md128_cut1.0e-10_it300_fix"]   # k=4 counterpart from Route C
@printf("%-4s %-9s %-8s %-12s %-12s %s\n","k","time(s)","niters","|θ0|","|θ1|","reason")
@printf("%-4d %-9.1f %-8d %-12.6f %-12.6f %s\n", 2, rF2.time, rF2.niters, abs(rF2.theta[1]), abs(rF2.theta[2]), rF2.reason)
@printf("%-4d %-9.1f %-8d %-12.6f %-12.6f %s\n", 4, rF4.time, rF4.niters, abs(rF4.theta[1]), abs(rF4.theta[2]), rF4.reason)

## Conclusion — the recommended configuration

Read the trade-offs off the tables above (filled by the run). The expected picture, to confirm with
the numbers:

- **Route A (maxdim).** The converged $\chi$ is well below 256, so a cap of ~96–128 should preserve
  $\lambda_0,\lambda_1$ to $\lesssim10^{-4}$ at a fraction of the cost — the largest single win.
- **Route B (ramp).** A `16→cap` schedule makes the early iterations cheap with no accuracy loss —
  stack it on top of Route A.
- **Route C (cutoff).** `1e-10` is essentially free accuracy-wise and trims the bond dim further;
  `1e-8` only if Route A/B aren't enough.
- **Route D (WII).** Cheaper per iteration (smaller temporal physical dim) but biases the absolute
  $\lambda$; keep VD2 for production unless the $p{=}0.1/p{=}0$ ratio is shown to survive WII.
- **Route E (warm start).** Same-$T$ reuse converges in a few iters — proof the seeds work; the
  cross-$T$ ladder warm start (with a `pad_tmps` helper) is the highest-payoff next step.
- **Route F (k).** $k=2$ is the cheaper default for the eigenvalue-only sweep; reserve $k=4$ for the
  entropy-profile sweep where clean de-mixing matters most.

**Recommended eigenvalue-only (Route 2) config:** `k=2`, `maxdim≈128` with a `maxdims=16→128` ramp,
`cutoff=1e-10`, VD2 — to be applied to the notebook-7 *lite* sweep. The entropy-profile sweep keeps
the accurate `k=4, maxdim=256, cutoff=1e-12` config (eigenvectors are more delicate than
eigenvalues).